### Deploying a Gradio App to Hugging Face Spaces

Turning a model into a *web app*

Two tools do this, and both are from the Hugging Face ecosystem:

- **Gradio** builds a web UI for an ML model from pure Python.
- **Hugging Face Spaces** is free hosting for those apps. A Space is just a Git repo on the Hub that HF builds and runs for you, giving your app a public URL.

We'll build a small **sentiment analysis** app (reusing the pipeline from the Sentiment Analysis demo), run it right here to test it, and then deploy it to a Space. The coding part happens in this notebook; the deployment part happens in the browser, and those steps are written out below as a checklist.

#### Build the app with Gradio

The core of Gradio is `gr.Interface`, which needs just three things:

- `fn` — the Python function to run (here, a function that classifies text)
- `inputs` — the input component(s) (a textbox)
- `outputs` — the output component(s) (a label with confidence scores)

Everything else (`title`, `description`, `examples`) is optional .

In [1]:
!pip install -q gradio transformers

In [2]:
import gradio as gr
from transformers import pipeline

# The model our app wraps.
classifier = pipeline("sentiment-analysis")

def classify(text):
    # top_k=None returns every label with its score, so gr.Label can show both.
    scores = classifier(text, top_k=None)

    return {s["label"]: float(s["score"]) for s in scores}

demo = gr.Interface(
    fn=classify,
    inputs=gr.Textbox(lines=3, placeholder="Type a sentence...", label="Your text"),
    outputs=gr.Label(label="Sentiment"),
    title="Sentiment Analysis",
    description="A simple sentiment classifier built with 🤗 Transformers and Gradio.",
    examples=[
        ["I absolutely loved this movie, it was fantastic!"],
        ["The service was slow and the food was cold."],
    ],
)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

#### Test it right here

`demo.launch()` starts the app. In Colab it renders **inline**, right under the cell, so you can try it immediately. (Passing `share=True` would also give you a temporary public link that lasts a few days — handy for a quick share, but Spaces is the permanent home.)

In [3]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://57d60fbd10d505509b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Type a sentence or click an example, hit **Submit**, and you'll see the predicted sentiment with confidence bars. That's a complete ML web app — now let's put it somewhere permanent.

#### Package it for a Space

A Gradio Space needs just two files at the repo root:

1. **`app.py`** — your Gradio code, ending in `demo.launch()`.
2. **`requirements.txt`** — the Python packages the Space should install. (You don't list `gradio` here — the Space provides it via the Gradio SDK. You only list your *extra* dependencies.)

The `%%writefile` magic below saves these as real files. If you're on Colab, you can find them in the file browser on the left after running the cells, then download them to upload to your Space.

In [6]:
%%writefile app.py
import spaces                       # MUST be first, before torch/transformers
import gradio as gr
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

@spaces.GPU
def classify(text):
    scores = classifier(text, top_k=None)
    return {s["label"]: float(s["score"]) for s in scores}

demo = gr.Interface(
    fn=classify,
    inputs=gr.Textbox(lines=3, placeholder="Type a sentence...", label="Your text"),
    outputs=gr.Label(label="Sentiment"),
    title="Sentiment Analysis",
    description="A simple sentiment classifier built with 🤗 Transformers and Gradio.",
    examples=[
        ["I absolutely loved this movie, it was fantastic!"],
        ["The service was slow and the food was cold."],
    ],
)

if __name__ == "__main__":
    demo.launch()

Overwriting app.py


In [5]:
%%writefile requirements.txt
transformers
torch

Writing requirements.txt


## Already deployed

https://huggingface.co/jinxxx123

#### Deploy to a Space — the browser steps

These are the UI steps to do once, in your browser. Nothing here needs code.

**1. Make sure you have a Hugging Face account.** Sign up (free) at https://huggingface.co/join and verify your email.

**2. Create a new Space.** Go to https://huggingface.co/new-space and fill in:
   - **Owner** — your username.
   - **Space name** — e.g. `sentiment-analysis-app` (this becomes part of the URL).
   - **Short description** — optional, one line.
   - **License** — optional (e.g. MIT).
   - **SDK** — choose **Gradio**.
   - **Hardware** — leave it on the free **CPU basic** tier; this model doesn't need a GPU.
   - **Visibility** — **Public** so others can open the link.

   Click **Create Space**.

**3. Note what HF created for you.** The new Space already contains a `README.md` with a YAML header (it sets `sdk: gradio`, `sdk_version`, `app_file: app.py`, etc.). Leave that file's YAML intact — it's what tells the Space how to launch. You can edit the prose below the YAML freely.

**4. Add your two files.** On the Space page, open the **Files** tab, then **+ Add file → Upload files**, and drag in `app.py` and `requirements.txt` (the ones you just wrote). Add a commit message and click **Commit changes**.
   - *Alternative:* **+ Add file → Create a new file**, name it `app.py`, and paste the code directly in the browser — no upload needed.

**5. Watch it build.** The Space status switches to **Building** while it installs `requirements.txt`, then to **Running**. First builds take a few minutes (it's installing PyTorch + Transformers), so if you're doing this live, create the Space *before* class.

**IMPORTANT NOTE: it takes ~2-3 minutes for a space to deploy.

**6. Open the App tab.** Once it says **Running**, the **App** tab shows your live sentiment classifier. The URL (`huggingface.co/spaces/your-username/sentiment-analysis-app`) is now shareable with anyone.
